# Data Quality — Шаг 2

Анализ и чистка данных из `data/raw/`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from agents.data_quality_agent import DataQualityAgent

raw_files = sorted(Path('../data/raw').glob('*.parquet'))
df = pd.read_parquet(raw_files[-1])
agent = DataQualityAgent()
print(f'Загружено: {df.shape}')

## Часть 1: Детектив

In [ ]:
# QualityReport
report = agent.detect_issues(df)
import json
print(json.dumps({k: v for k, v in report.items() if k != 'outliers'}, indent=2, ensure_ascii=False))

In [ ]:
# Визуализация пропусков
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

missing_pct = df.isna().mean() * 100
missing_pct[missing_pct > 0].plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Пропущенные значения (%)')
axes[0].set_ylabel('%')

if 'label' in df.columns:
    df['label'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
    axes[1].set_title('Дисбаланс классов')
    axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Часть 2: Хирург

In [ ]:
# Стратегия A: мягкая чистка
strategy_A = {
    'missing': 'median',
    'duplicates': 'drop',
    'outliers': 'clip_iqr',
    'imbalance': 'none',
    'text_quality': 'drop_empty'
}
df_clean_A = agent.fix(df, strategy=strategy_A)
print(f'Стратегия A: {len(df)} → {len(df_clean_A)} строк')

In [ ]:
# Стратегия B: жёсткая чистка
strategy_B = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'drop',
    'imbalance': 'undersample',
    'text_quality': 'drop_empty'
}
df_clean_B = agent.fix(df, strategy=strategy_B)
print(f'Стратегия B: {len(df)} → {len(df_clean_B)} строк')

In [ ]:
# Сравнение стратегий
print('=== Сравнение: Стратегия A ===')
cmp_A = agent.compare(df, df_clean_A)
print('\n=== Сравнение: Стратегия B ===')
cmp_B = agent.compare(df, df_clean_B)

## Часть 3: Аргумент

**Рекомендация:** Для задачи news classification предпочтительна **Стратегия A** (мягкая чистка):

- AG News хорошо сбалансирован (по 4 класса примерно равные) — балансировка не нужна
- Тексты новостей редко содержат пропуски — `median` сохраняет больше данных чем `drop`
- Объём данных важен для качества TF-IDF признаков
- Компромисс: немного меньше "чистоты" = значительно больше данных для обучения